# 📉 Customer Churn Analysis
> *Identify patterns behind subscription cancellations — and fix them.*

**What this notebook covers:**
1. Load & explore raw customer data
2. Engineer behavioral features (engagement score, LTV, at-risk flag)
3. Compute headline churn metrics
4. Segment analysis — churn by plan, contract, tenure
5. Identify key churn drivers via correlation
6. Cohort retention analysis
7. Generate prioritised retention recommendations
8. Export results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, warnings
sys.path.append('../scripts')
from churn_analysis import *
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='Set2')
print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
df_raw = pd.read_csv('../data/raw/customers_raw.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Class balance
print(df_raw['churned'].value_counts())
print(f"\nOverall churn rate: {df_raw['churned'].mean()*100:.1f}%")

## 2. Feature Engineering

In [ ]:
df = engineer_features(df_raw.copy())
df[['customer_id','engagement_score','ltv_estimate','tenure_band','nps_segment','at_risk']].head(10)

## 3. Headline Metrics

In [ ]:
metrics = compute_churn_metrics(df)
for k, v in metrics.items():
    print(f'  {k:<35} {v}')

## 4. Engagement: Churned vs Retained

In [ ]:
engagement = engagement_analysis(df)
eng_df = pd.DataFrame([
    {'Metric': k.replace('avg_','').replace('_churned','').replace('_retained',''),
     'Group': 'Churned' if 'churned' in k else 'Retained',
     'Value': v}
    for k, v in engagement.items()
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Engagement score distribution
for label, grp in df.groupby('churned'):
    axes[0].hist(grp['engagement_score'], alpha=0.6, bins=15,
                 label='Churned' if label==1 else 'Retained')
axes[0].set_title('Engagement Score Distribution')
axes[0].set_xlabel('Engagement Score')
axes[0].legend()

# Support calls
df.groupby('churned')['support_calls'].mean().plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c'])
axes[1].set_title('Avg Support Calls: Retained vs Churned')
axes[1].set_xticklabels(['Retained','Churned'], rotation=0)
axes[1].set_ylabel('Avg Support Calls')

plt.tight_layout()
plt.savefig('../outputs/engagement_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Churn by Segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['plan', 'contract_type', 'tenure_band']):
    seg = churn_by_segment(df, col)
    ax.barh(seg[col].astype(str), seg['churn_rate_pct'], color='#e74c3c')
    ax.set_xlabel('Churn Rate (%)')
    ax.set_title(f'Churn by {col.replace("_"," ").title()}')
    for i, v in enumerate(seg['churn_rate_pct']):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/churn_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Churn Drivers

In [ ]:
drivers = rank_churn_drivers(df)
display(drivers)

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in drivers['correlation_with_churn']]
plt.barh(drivers['feature'], drivers['correlation_with_churn'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Churn (red = increases churn risk)')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../outputs/churn_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cohort Retention

In [ ]:
cohort = cohort_retention(df)
display(cohort)

plt.figure(figsize=(10, 4))
plt.bar(cohort['tenure_band'].astype(str), cohort['retention_rate'], color='#3498db')
plt.ylim(0, 110)
plt.title('Retention Rate by Tenure Cohort')
plt.xlabel('Tenure Band')
plt.ylabel('Retention Rate (%)')
for i, v in enumerate(cohort['retention_rate']):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Retention Recommendations

In [ ]:
engagement_data = engagement_analysis(df)
recs = generate_recommendations(metrics, engagement_data, drivers)

for i, r in enumerate(recs, 1):
    print(f"[{r['priority']}] #{i} — {r['segment']}")
    print(f"  Finding : {r['finding']}")
    print(f"  Action  : {r['action']}")
    print()

## 9. Export At-Risk Customers

In [ ]:
at_risk = df[(df['at_risk'] == 1) & (df['churned'] == 0)]
print(f'At-risk customers identified: {len(at_risk)}')
at_risk.to_csv('../outputs/at_risk_customers.csv', index=False)
df.to_csv('../data/processed/customers_enriched.csv', index=False)
print('Exported ✓')
at_risk[['customer_id','plan','tenure_months','engagement_score','support_calls','monthly_charge']]